# Chapter 14: Consensus
*From: Database Internals*

## TL;DR

- **Consensus** is the problem of getting a set of processes to agree on a single value, even when some of them crash, messages get delayed, or (in the Byzantine case) some actively lie.
- The chapter walks the algorithm family tree: **atomic broadcast** (ZAB) — the primitive that underpins consensus; **Classic Paxos** — the canonical asynchronous single-decree protocol; **Multi-Paxos / Fast Paxos / EPaxos / Flexible Paxos** — optimizations over classic; **Raft** — an understandability-first redesign; and **PBFT** — consensus under adversarial failures.
- Crash-tolerant consensus needs `n = 2f + 1` nodes to tolerate `f` failures. Byzantine consensus needs `n = 3f + 1` and cross-validation of every step, pushing message complexity to O(N^2).
- **FLP impossibility** says no deterministic consensus is guaranteed to terminate in a fully asynchronous model; real algorithms guarantee **safety** always and **liveness** via partial synchrony + failure detectors.
- Strong-leader approaches (Multi-Paxos, Raft, ZAB) win on throughput; leaderless / dependency-graph approaches (EPaxos) win on availability and WAN latency.

---

## Problem Statement

Consensus is characterized by three theoretical properties:

- **Agreement** — every correct process decides on the same value. No two correct processes may disagree on the outcome.
- **Validity** — the decided value must have been proposed by one of the processes. Agreeing on a hardcoded default does not count.
- **Termination** — every correct process eventually decides. The algorithm must not hang indefinitely.

Related operational requirements a real consensus system must satisfy:

- **Safety under any number of failures** — inconsistent or incorrect state must never be observable, even under arbitrary message loss/reordering/duplication.
- **Liveness under bounded failures** — progress must continue as long as a majority of nodes are up and can communicate (partial synchrony).
- **Totally ordered log** — for state-machine replication, commands must be applied in the same order on every replica.
- **Linearizable reads** — in Multi-Paxos / Raft style systems, reads must reflect the most recent committed write (often via leader leases).
- **Recovery** — a newly joined or recovering node must be able to catch up to the committed prefix of the log.

Impossibility & trade-off backdrop:

- **FLP** — in a fully asynchronous model with even one crash failure, no deterministic protocol can guarantee all three properties simultaneously. Real systems sidestep FLP via randomized timers (Raft), failure detectors, or weak synchrony assumptions.
- **CAP pressure** — on a network partition, a consensus quorum side keeps serving; the minority side loses availability to preserve consistency.

---

## Back-of-Envelope: Quorums, Round-Trips, Message Complexity

Before picking an algorithm, reason about **how many nodes** you need, **how many round-trips** a decision takes, and **how many messages** cross the wire per decision.

### Quorum math: crash (2f+1) vs Byzantine (3f+1)

In [ ]:
# How many replicas do I need to tolerate f failures?
def crash_nodes(f):
    # Any two majorities intersect -> safety
    return 2 * f + 1

def byzantine_nodes(f):
    # Must outnumber (faulty + unresponsive-but-honest) replicas
    return 3 * f + 1

print(f"{'f':>3} {'crash (2f+1)':>14} {'byzantine (3f+1)':>18}")
for f in range(0, 6):
    print(f"{f:>3} {crash_nodes(f):>14} {byzantine_nodes(f):>18}")

### Quorum sizes per algorithm

In [ ]:
# Quorum = minimum votes needed to proceed in a phase.
# For Paxos/Raft that's a simple majority; Fast Paxos and EPaxos
# change the math to cut a round-trip.
from math import ceil

def majority(n):
    return n // 2 + 1

def fast_paxos_quorum(f):
    # Fast Paxos: requires 3f+1 total acceptors and a fast quorum of 2f+1
    # (vs f+1 in classic Paxos). See [JUNQUEIRA07] / [LAMPORT06].
    total = 3 * f + 1
    fast_q = 2 * f + 1
    return total, fast_q

def epaxos_quorums(f):
    # EPaxos, per the chapter: fast quorum is ceil(3f/4); slow quorum is
    # floor(f/2) + 1 (used when dependency sets disagree).
    fast = ceil(3 * f / 4)
    slow = f // 2 + 1
    return fast, slow

n = 5
f = 2
print(f"n = {n}, f = {f}")
print(f"  Classic / Multi-Paxos majority quorum : {majority(n)}")
total_fp, fast_fp = fast_paxos_quorum(f)
print(f"  Fast Paxos total / fast quorum        : {total_fp} / {fast_fp}")
fast, slow = epaxos_quorums(f)
print(f"  EPaxos fast / slow quorum             : {fast} / {slow}")
print(f"  PBFT total replicas for f={f}         : {3*f + 1} (needs 2f+1={2*f+1} matching Commits)")


### Round-trip latency model

In [ ]:
# One-way WAN RTT between replicas (in ms). Commit latency dominates
# user-visible write latency on the critical path.
RTT = 50  # ms (e.g., cross-region US-EU one-way ~ 40-70ms)

algos = [
    # (name, round-trips on happy path)
    ("Classic Paxos",   2),   # Prepare/Promise + Accept/Accepted
    ("Multi-Paxos",     1),   # skip Prepare; leader already chosen
    ("Fast Paxos",      1),   # client -> acceptors directly
    ("EPaxos (fast)",   1),   # PreAccept with agreement
    ("EPaxos (slow)",   2),   # PreAccept + Accept on conflict
    ("Flexible Paxos",  1),   # same as Multi-Paxos once leader is up
    ("Raft",            1),   # AppendEntries + Ack; commit notified next beat
    ("PBFT",            3),   # Pre-prepare, Prepare, Commit
]

print(f"{'Algorithm':<18} {'Round-trips':>12} {'Commit latency (2*RTT*rt)':>28}")
for name, rt in algos:
    print(f"{name:<18} {rt:>12} {2*RTT*rt:>24} ms")

### Message complexity per decision

In [ ]:
# Rough per-decision message counts on the happy path.
# N = total replicas, Q = quorum size.
N = 5
Q = majority(N)   # 3

def leader_based_msgs(N, rounds):
    # Leader -> N-1 followers, then N-1 acks back. Repeated per round.
    return rounds * 2 * (N - 1)

def byzantine_msgs(N):
    # Pre-prepare (N-1) + Prepare all-to-all (N*(N-1)) + Commit all-to-all (N*(N-1))
    return (N - 1) + 2 * N * (N - 1)

print(f"N = {N}")
print(f"  Multi-Paxos / Raft  : {leader_based_msgs(N, 1)} msgs/decision (1 round)")
print(f"  Classic Paxos       : {leader_based_msgs(N, 2)} msgs/decision (2 rounds)")
print(f"  PBFT                : {byzantine_msgs(N)} msgs/decision")
# Note: PBFT grows O(N^2) so scaling to 100+ nodes is impractical without
# aggregation or threshold signatures.

### Availability under f failures

In [ ]:
# What fraction of time is the cluster writable, given per-node uptime?
def cluster_write_availability(n, per_node_uptime):
    """Prob(at least majority nodes up) for independent node failures."""
    from math import comb
    q = majority(n)
    p = per_node_uptime
    avail = 0.0
    for k in range(q, n + 1):
        avail += comb(n, k) * (p ** k) * ((1 - p) ** (n - k))
    return avail

for n in (3, 5, 7):
    a = cluster_write_availability(n, 0.99)
    print(f"n={n} nodes @ 99% uptime each -> cluster write avail = {a*100:.4f}%")

---

## High-Level Landscape

```mermaid
graph TD
    BCAST[Broadcast primitives]
    BCAST --> BE[Best-effort broadcast]
    BCAST --> REL[Reliable broadcast flooding N^2]
    BCAST --> ATOMIC[Atomic broadcast reliable + total order]
    ATOMIC --> VS[Virtual Synchrony dynamic groups]
    ATOMIC --> ZAB[ZAB - Zookeeper]
    ATOMIC --> CONS[Consensus]

    CONS --> PAXOS[Classic Paxos single-decree]
    PAXOS --> MP[Multi-Paxos leader reuse]
    PAXOS --> FP[Fast Paxos 1 round-trip]
    PAXOS --> EP[EPaxos dependency graph]
    PAXOS --> FLEX[Flexible Paxos Q1 + Q2 > N]
    PAXOS --> RAFT[Raft strong leader]

    CONS --> BFT[Byzantine consensus]
    BFT --> PBFT[PBFT 3-phase cross-validate]
```

---

## Deep Dive 1: Atomic Broadcast and Virtual Synchrony

**Why it matters:** consensus in an asynchronous crash-failure model is equivalent to atomic broadcast. Understand the broadcast primitive and the consensus primitive at the same time.

**Three levels of broadcast guarantees:**

| Primitive | Reliability | Ordering | Cost |
|---|---|---|---|
| Best-effort | Sender-only; fails silently on sender crash | None | O(N) |
| Reliable (flooding) | Every correct process eventually sees every message | None | O(N^2) |
| Atomic / total-order multicast | Reliable + all correct processes deliver in identical order | Varies; often uses a sequencer |

**Virtual synchrony** extends atomic broadcast to *dynamic* groups: membership changes are themselves broadcast as "view changes", and a message sent in view V is delivered only in view V. This creates a barrier that messages cannot cross, giving atomic delivery within a view.

```mermaid
sequenceDiagram
    participant C as Coordinator
    participant P1 as Process 1
    participant P2 as Process 2
    participant P3 as Process 3

    Note over C,P3: Best-effort: C crashes mid-send -> P3 never sees msg
    C->>P1: msg
    C->>P2: msg
    C--xP3: (crash)

    Note over C,P3: Reliable / flooding: each receiver forwards
    P1->>P2: msg (forward)
    P1->>P3: msg (forward)
    P2->>P3: msg (forward)

    Note over C,P3: Atomic: flood + sequencer assigns order
    P1->>P2: msg seq=42
    P2->>P3: msg seq=42
```

---

## Deep Dive 2: Zookeeper Atomic Broadcast (ZAB)

Three-phase, leader-based protocol used by Apache Zookeeper. Epochs are monotonic — at most one leader per epoch — and the protocol transitions between phases explicitly on leader failure.

**Phases:**
1. **Discovery** — prospective leader learns the highest epoch known by any follower and proposes `epoch + 1`. After this point, followers refuse proposals from earlier epochs.
2. **Synchronization** — leader brings followers up-to-date by replaying committed proposals from previous epochs, ensuring a consistent log prefix. Once followers ack, the leader is "established".
3. **Broadcast** — the active phase. Leader receives client writes, sequences them, sends `Proposal`, collects a quorum of acks, then sends `Commit`. Continues until the leader crashes or is suspected.

```mermaid
sequenceDiagram
    participant L as Leader (prospective)
    participant F1 as Follower 1
    participant F2 as Follower 2

    rect rgb(240,240,255)
    Note over L,F2: Discovery
    L->>F1: NEWEPOCH(e+1)
    L->>F2: NEWEPOCH(e+1)
    F1-->>L: ACK + lastZxid
    F2-->>L: ACK + lastZxid
    end

    rect rgb(240,255,240)
    Note over L,F2: Synchronization
    L->>F1: NEWLEADER + replay committed txns
    L->>F2: NEWLEADER + replay committed txns
    F1-->>L: ACK
    F2-->>L: ACK
    end

    rect rgb(255,250,240)
    Note over L,F2: Broadcast (steady state)
    L->>F1: Proposal(txn)
    L->>F2: Proposal(txn)
    F1-->>L: Ack
    F2-->>L: Ack
    L->>F1: Commit
    L->>F2: Commit
    end
```

Recovery is cheap: followers respond with their highest committed proposal, and the leader only needs to stream deltas from the most up-to-date replica.

---

## Deep Dive 3: Classic Paxos

Single-decree Paxos decides *one* value. Participants play three (overlapping) roles: **proposers**, **acceptors**, **learners**. Proposal numbers are globally unique and monotonically increasing (typically `(ts, node_id)`).

**Two phases:**
- **Phase 1 (Prepare / Promise)** — proposer picks proposal number `n` and sends `Prepare(n)` to a majority of acceptors. Each acceptor: if `n` beats any prepare it has seen, it promises not to accept anything lower and reports the highest-numbered value (if any) it has previously accepted.
- **Phase 2 (Accept / Accepted)** — proposer picks `v` = the highest-numbered value from the Promises (or its own if none), sends `Accept!(n, v)` to acceptors. Acceptors accept unless they have since promised to a higher `n`.

```mermaid
sequenceDiagram
    participant P as Proposer
    participant A1 as Acceptor 1
    participant A2 as Acceptor 2
    participant A3 as Acceptor 3
    participant L as Learner

    Note over P,A3: Phase 1: Prepare / Promise
    P->>A1: Prepare(n)
    P->>A2: Prepare(n)
    P->>A3: Prepare(n)
    A1-->>P: Promise(n, prev_v?)
    A2-->>P: Promise(n, prev_v?)

    Note over P,A3: Phase 2: Accept / Accepted
    P->>A1: Accept!(n, v)
    P->>A2: Accept!(n, v)
    P->>A3: Accept!(n, v)
    A1-->>L: Accepted(n, v)
    A2-->>L: Accepted(n, v)
```

**Safety intuition:** any two majority quorums intersect in at least one node. If a value was accepted by a majority, any subsequent Prepare with a higher `n` will observe it via at least one acceptor in its Promise quorum, and will be forced to re-propose the same value.

**Failure scenarios (why it works under proposer / acceptor failure):**

```mermaid
sequenceDiagram
    participant P1 as Proposer 1
    participant P2 as Proposer 2
    participant A1 as Acceptor 1
    participant A2 as Acceptor 2
    participant A3 as Acceptor 3

    Note over P1,A3: Scenario: P1 fails after A1 accepts v1
    P1->>A1: Accept!(1, v1)
    A1-->>P1: Accepted
    Note over P1: P1 crashes

    Note over P2,A3: P2 starts with n=2, picks up v1 via A1
    P2->>A1: Prepare(2)
    P2->>A2: Prepare(2)
    A1-->>P2: Promise(2, (1, v1))
    A2-->>P2: Promise(2, none)
    P2->>A1: Accept!(2, v1)
    P2->>A2: Accept!(2, v1)
    P2->>A3: Accept!(2, v1)
```

**Liveness hazard:** dueling proposers can live-lock — each keeps bumping `n` and poisoning the other's promises. Fix: randomized exponential backoff (same idea as Raft election timers).

---

## Deep Dive 4: Multi-Paxos and Paxos Variants

### Multi-Paxos
Classic Paxos needs a Prepare round *per decision*. Multi-Paxos elects a **distinguished proposer (leader)** once, then that leader runs only Phase 2 for subsequent decisions. Effect: steady-state commit drops to a single round-trip, identical to ZAB/Raft. A **leader lease** allows linearizable reads without a full Paxos round — but depends on bounded clock drift between replicas.

### Fast Paxos
Lets any proposer (or client) contact acceptors directly, skipping the leader on the happy path. Cost: quorum sizes grow from `f+1` to `2f+1` out of `3f+1` total, and *collisions* (different values accepted by different acceptors) force a fallback to classic-Paxos recovery.

### EPaxos (Egalitarian Paxos)
No fixed leader; every command gets a per-command leader. Commands carry **dependencies** (other commands that interfere with them). The leader pre-accepts with a fast quorum; if replies agree on dependencies, commit fast-path; otherwise run a slow-path `Accept` round with updated deps.

```mermaid
sequenceDiagram
    participant C as Client
    participant P1 as Replica 1 (leader for cmd)
    participant P2 as Replica 2
    participant P3 as Replica 3
    participant P5 as Replica 5

    Note over C,P5: Fast path - dependencies agree
    C->>P1: propose cmd A
    P1->>P2: PreAccept(A, deps=empty)
    P1->>P3: PreAccept(A, deps=empty)
    P2-->>P1: Ack(deps=empty)
    P3-->>P1: Ack(deps=empty)
    P1->>C: Committed A
    P1->>P2: Commit A (async)

    Note over C,P5: Slow path - conflict detected
    C->>P5: propose cmd B
    P5->>P3: PreAccept(B, deps=empty)
    P3-->>P5: Nack deps={A}
    P5->>P3: Accept(B, deps={A})
    P3-->>P5: Ack
    P5->>C: Committed B after A
```

Execution at each replica builds the dependency DAG and executes in reverse-dependency order. Interference-free commands commit in parallel with no coordination — excellent for geo-distributed workloads.

### Flexible Paxos
Insight: only the **first-phase quorum Q1** must intersect the **second-phase quorum Q2**, not every pair of quorums. So any `Q1 + Q2 > N` works. With `N=5`, set `Q1=4`, `Q2=2`: elections get harder but the hot-path commit only needs 2 acks. This trades availability (harder to elect a new leader) for latency / throughput once a stable leader exists.

---

## Deep Dive 5: Raft

Raft's design goal is *understandability*. It splits consensus into three sub-problems and solves each with a single, strongly-defined mechanism:

1. **Leader election** — at most one leader per monotonic `term`. A follower whose election timeout expires becomes a candidate, increments term, and requests votes.
2. **Log replication** — leader appends commands and ships `AppendEntries` RPCs with the `(prevLogIndex, prevLogTerm)` of the entry immediately before. Followers reject mismatched logs, prompting the leader to back up and retry.
3. **Safety** — only candidates with logs at least as up-to-date as a voter's log can win that vote. This preserves the "leader has all committed entries" invariant across term changes.

**Roles:** follower (default, passive), candidate (running election), leader (one per term).

```mermaid
sequenceDiagram
    participant C as Candidate (P1)
    participant F2 as Follower 2
    participant F3 as Follower 3

    Note over C,F3: Leader election
    C->>F2: RequestVote(term=t, lastLogIdx, lastLogTerm)
    C->>F3: RequestVote(term=t, lastLogIdx, lastLogTerm)
    F2-->>C: Vote granted
    F3-->>C: Vote granted
    Note over C: Becomes Leader for term t

    Note over C,F3: Log replication (steady state)
    C->>F2: AppendEntries(prevIdx, prevTerm, [entry])
    C->>F3: AppendEntries(prevIdx, prevTerm, [entry])
    F2-->>C: Ack
    F3-->>C: Ack
    Note over C: Commit locally
    C->>F2: AppendEntries(commitIdx++)
    C->>F3: AppendEntries(commitIdx++)
```

**Commit flow:** (a) leader appends to its own log, (b) replicates to majority, (c) commits locally, (d) propagates commit index via next heartbeat. Followers apply committed entries to their state machine in log order.

**Split-vote handling:** randomized election timeouts (typically 150–300 ms) ensure candidates usually don't start simultaneously. On a split vote, everyone goes back to follower and re-randomizes.

**Log-divergence repair:** when a new leader takes over, it finds the longest common prefix with each follower's log and overwrites the follower's tail. Its own log is append-only.

---

## Deep Dive 6: Byzantine Consensus and PBFT

When nodes may be malicious (lie, forge, collude), crash-tolerance is not enough. **PBFT (Practical Byzantine Fault Tolerance)** assumes weak synchrony and authenticated encrypted channels. Quorum size jumps to `2f + 1` out of `3f + 1`: you must outvote the `f` malicious replicas plus the `f` honest-but-slow replicas.

**Three-phase protocol (one per decision):**

```mermaid
sequenceDiagram
    participant C as Client
    participant P as Primary (P1)
    participant B2 as Backup 2
    participant B3 as Backup 3
    participant B4 as Backup 4 (faulty)

    C->>P: Request(op)
    P->>B2: Pre-prepare(view, n, digest, op)
    P->>B3: Pre-prepare(view, n, digest, op)
    P->>B4: Pre-prepare(view, n, digest, op)

    Note over B2,B4: Prepare phase - all-to-all
    B2->>P: Prepare(view, n, digest)
    B2->>B3: Prepare(view, n, digest)
    B3->>P: Prepare(view, n, digest)
    B3->>B2: Prepare(view, n, digest)
    Note over B2: Collect 2f matching Prepares

    Note over B2,B4: Commit phase - all-to-all
    B2->>P: Commit(view, n, digest)
    B2->>B3: Commit(view, n, digest)
    B3->>B2: Commit(view, n, digest)
    Note over B2: Collect 2f+1 matching Commits

    B2-->>C: Reply(result)
    B3-->>C: Reply(result)
    Note over C: Accepts after f+1 matching replies
```

**Key mechanisms:**
- **Digests + signatures** — after pre-prepare, replicas exchange only the digest (not the payload), each signed by sender. Cross-validates identity and integrity without rebroadcasting payloads.
- **View changes** — if the primary misbehaves, backups stop responding to it and broadcast a view-change. When the next primary collects `2f` view-change messages, it starts a new view.
- **Stable checkpoints** — every N requests, primary broadcasts a digest of state; `2f+1` matching responses form a checkpoint proof, after which earlier pre-prepare/prepare/commit messages can be garbage-collected.
- **Read-only fast path** — reads broadcast to all replicas; client accepts after `2f+1` matching responses (one round-trip, no three-phase).

Because each of prepare/commit is all-to-all, PBFT is O(N^2) per decision. This is why permissioned blockchains (Hyperledger, Tendermint) top out at ~100 validators without extra techniques like threshold signatures or BLS aggregation.

---

## Trade-off Matrix

| Algorithm | Nodes to tolerate f | Phase-2 quorum | Round-trips (happy) | Leader model | Failure model | Strength | Weakness |
|---|---|---|---|---|---|---|---|
| **Classic Paxos** | 2f+1 | f+1 (majority) | 2 | Per-decision proposer | Crash-async | Proven safe under any failure pattern | Slow; dueling proposers can live-lock |
| **Multi-Paxos** | 2f+1 | f+1 | 1 | Long-lived leader | Crash-async | High throughput; leader lease enables fast linearizable reads | Leader is a bottleneck; elections disrupt traffic |
| **Fast Paxos** | 3f+1 | 2f+1 | 1 | Coordinator + direct clients | Crash-async | Saves a round-trip on low-contention writes | Larger quorum; collisions trigger expensive fallback |
| **EPaxos** | 2f+1 | ceil(3f/4) fast / f+1 slow | 1 (fast) / 2 (slow) | Per-command leader | Crash-async | No hot leader; great WAN latency for non-conflicting cmds | Complex execution via dependency DAG |
| **Flexible Paxos** | 2f+1 | configurable, Q1+Q2 > N | 1 | Long-lived leader | Crash-async | Tunes availability vs latency; shrinks accept quorum | Fewer tolerated failures during leader elections |
| **Raft** | 2f+1 | f+1 | 1 | Long-lived leader | Crash-async | Understandable; strong invariants; wide adoption | Leader-centric; leader elections stall writes |
| **PBFT** | 3f+1 | 2f+1 | 3 | Primary + all-to-all | Byzantine | Tolerates malicious nodes | O(N^2) msgs; doesn't scale past ~100 replicas |

### When to use which

- **Raft** — default choice for new systems needing crash-tolerant consensus (etcd, Consul, CockroachDB, TiKV). Pick it unless you have a specific reason not to.
- **Multi-Paxos** — legacy, or you specifically need Chubby/Spanner-style subtleties (ballot numbers, flexible quorums from day one).
- **EPaxos** — geo-distributed, read-heavy, low-conflict workloads where you want to avoid a single-leader bottleneck.
- **Fast Paxos** — rare in production; the collision cost usually eats the latency savings.
- **Flexible Paxos** — when you need to tune latency/availability asymmetry (e.g., write-heavy datacenter with cheap leader election).
- **ZAB** — you're running Zookeeper.
- **PBFT (or modern BFT descendants like HotStuff, Tendermint)** — permissioned blockchain, adversarial multi-party systems, or regulated settings where Byzantine assumptions are warranted.

---

## Key Takeaways

1. **Consensus = Agreement + Validity + Termination.** FLP says you can't guarantee all three in a fully async crash model — real systems pick partial synchrony + failure detectors to gain liveness while preserving safety.
2. **Atomic broadcast and consensus are equivalent** in an async crash model. ZAB and Multi-Paxos look almost identical for a reason.
3. **Quorum intersection is the safety lever.** Classic Paxos/Raft need a majority because any two majorities overlap. Flexible Paxos generalizes this: only `Q1 + Q2 > N` is required, letting you trade election cost against commit latency.
4. **Byzantine tolerance is 1.5× more expensive in nodes and square in messages.** You need `3f+1` replicas (not `2f+1`) and O(N^2) messages per decision because every phase must be cross-validated.
5. **A stable leader is the single biggest throughput optimization.** Multi-Paxos, Raft, ZAB all converge on "skip the expensive phase-1 once you have a leader you trust for a while".
6. **Leaderless / per-command-leader designs (EPaxos) trade complexity for availability.** They shine on WAN + non-conflicting workloads but are much harder to implement correctly.
7. **Understandability matters in production.** Raft's broad adoption vs Paxos's niche uptake shows that if engineers can't reason about the protocol, they'll misuse it — even if the alternative is formally equivalent.

---

## See Also

- [[01-atomic-broadcast-and-virtual-synchrony]]
- [[02-zookeeper-atomic-broadcast-zab]]
- [[03-classic-paxos]]
- [[04-multi-paxos-and-variants]]
- [[05-raft-consensus]]
- [[06-byzantine-consensus-pbft]]